# Lab 17: A2C on CartPole

**Module 17** | Read `notes/17-policy-gradients-actor-critic.pdf` before running this notebook.

Train an A2C actor-critic agent on CartPole-v1 and compare learning speed to the DQN lab.

Run every cell top to bottom. Code is complete, no fill-in sections. Markdown cells explain what each block does and why.


## Runtime device

PyTorch can run on your NVIDIA GPU through CUDA or fall back to CPU. GPU execution moves matrix operations off the host and typically speeds up neural network training by a large factor. The next cell detects what is available and prints the active device so you know whether to expect fast training or should keep batch sizes small.


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"CUDA enabled, {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("CUDA not available, using CPU. Labs still run; training may take longer.")


## Actor-Critic (A2C) on CartPole

**A2C** (Advantage Actor-Critic) is an on-policy algorithm that learns both a policy (actor) and a value function (critic).
Unlike DQN, it optimizes the policy directly with policy gradients rather than estimating Q-values for every action.

Advantages of actor-critic methods on CartPole:

- Often learn smoothly on control tasks with continuous state spaces.
- No replay buffer; updates use the current policy's rollouts.
- The critic reduces variance of policy-gradient estimates.

You will train A2C for **30,000 timesteps** on the same CartPole-v1 task used in Lab 16, then compare final performance to DQN expectations.


### Environment setup

We use the same `Monitor` wrapper and episode-printing callback as Lab 16 so you can compare training dynamics side by side.


In [ ]:
import gymnasium as gym
import numpy as np
from pathlib import Path
from stable_baselines3 import A2C
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.monitor import Monitor

ROOT = Path("..").resolve()
ACTION_NAMES = {0: "left", 1: "right"}
SOLVED_THRESHOLD = 195


class EpisodeStatsCallback(BaseCallback):
    def __init__(self, print_every: int = 5):
        super().__init__()
        self.print_every = print_every
        self.episode_count = 0

    def _on_step(self) -> bool:
        for info in self.locals.get("infos", []):
            if info and "episode" in info:
                self.episode_count += 1
                ep = info["episode"]
                if self.episode_count % self.print_every == 0:
                    print(
                        f"  Training episode {self.episode_count}: "
                        f"reward={ep['r']:.0f}, length={ep['l']}"
                    )
        return True


env = Monitor(gym.make("CartPole-v1"))
print("CartPole-v1 environment ready for A2C")
print(f"  Actions: 0=left, 1=right")
print(f"  Solved threshold: average reward >= {SOLVED_THRESHOLD} over 100 episodes")


### Train the A2C agent

A2C updates the actor and critic together from on-policy rollouts collected during the same 30,000-step budget used in Lab 16.


In [ ]:
TOTAL_TIMESTEPS = 30_000

model = A2C("MlpPolicy", env, verbose=1)
stats_callback = EpisodeStatsCallback(print_every=5)

print(f"Starting A2C training for {TOTAL_TIMESTEPS:,} timesteps...")
model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=stats_callback)
print("Training finished.")


### Evaluate and compare with DQN expectations

We print detailed training statistics, plot rewards, run 5 greedy test episodes, and display a comparison table against typical Lab 16 (DQN) outcomes.


In [ ]:
import matplotlib.pyplot as plt

rewards = env.get_episode_rewards()
lengths = env.get_episode_lengths()

print("=" * 60)
print("A2C TRAINING SUMMARY")
print("=" * 60)
print(f"Episodes completed: {len(rewards)}")
print(f"Min / max reward: {min(rewards):.0f} / {max(rewards):.0f}")
print(f"Overall mean reward: {np.mean(rewards):.2f}")
print(f"Overall mean episode length: {np.mean(lengths):.1f} steps")
if len(rewards) >= 10:
    mean_last_10 = float(np.mean(rewards[-10:]))
    print(f"Mean reward (last 10): {mean_last_10:.2f}")
if len(rewards) >= 100:
    mean_last_100 = float(np.mean(rewards[-100:]))
    print(f"Mean reward (last 100): {mean_last_100:.2f}")

plt.figure(figsize=(9, 4))
plt.plot(rewards, alpha=0.6, label="Episode reward")
if len(rewards) >= 10:
    window = np.convolve(rewards, np.ones(10) / 10, mode="valid")
    plt.plot(range(9, 9 + len(window)), window, linewidth=2, label="10-episode moving avg")
plt.axhline(SOLVED_THRESHOLD, color="gray", linestyle="--", label=f"solved threshold ({SOLVED_THRESHOLD})")
plt.xlabel("Episode")
plt.ylabel("Total reward")
plt.title("A2C on CartPole-v1: training rewards")
plt.legend()
plt.tight_layout()
plt.show()

print()
print("=" * 60)
print("5 GREEDY TEST EPISODES")
print("=" * 60)

test_env = gym.make("CartPole-v1")
test_totals = []

for ep in range(5):
    obs, _ = test_env.reset()
    done = False
    step = 0
    total_reward = 0.0
    actions_taken: list[int] = []

    while not done:
        action, _ = model.predict(obs, deterministic=True)
        action_int = int(action)
        actions_taken.append(action_int)
        obs, reward, terminated, truncated, _ = test_env.step(action_int)
        total_reward += reward
        step += 1
        done = terminated or truncated

    test_totals.append(total_reward)
    preview = ", ".join(ACTION_NAMES[a] for a in actions_taken[:10])
    if len(actions_taken) > 10:
        preview += ", ..."
    print(f"Episode {ep + 1}: steps={step}, total_reward={total_reward:.0f}, actions={preview}")

test_env.close()
env.close()

mean_last_10 = float(np.mean(rewards[-10:])) if len(rewards) >= 10 else float(np.mean(rewards))
test_mean = float(np.mean(test_totals))

print()
print("=" * 60)
print("A2C vs DQN (LAB 16) COMPARISON")
print("=" * 60)
rows = [
    ("Algorithm", "A2C (this lab)", "DQN (Lab 16)"),
    ("Learning style", "On-policy rollouts", "Off-policy + replay buffer"),
    ("What is learned", "Policy + value function", "Q-function, then greedy policy"),
    ("Typical mean reward (last 10)", f"{mean_last_10:.1f}", "400 to 500 (expected)"),
    ("Your test mean (5 episodes)", f"{test_mean:.1f}", "similar order of magnitude"),
    ("Sample efficiency", "Discards old rollouts", "Reuses past transitions"),
]
col_w = [28, 22, 28]
header = rows[0]
print(f"{header[0]:<{col_w[0]}} | {header[1]:<{col_w[1]}} | {header[2]:<{col_w[2]}}")
print("-" * (sum(col_w) + 6))
for row in rows[1:]:
    print(f"{row[0]:<{col_w[0]}} | {row[1]:<{col_w[1]}} | {row[2]:<{col_w[2]}}")

print()
print("On CartPole both families usually converge; differences show more on harder environments.")
print(f"Solved threshold reminder: average reward >= {SOLVED_THRESHOLD} over 100 episodes.")


## Interpretation

CartPole is easy enough that DQN and A2C often reach similar final performance when trained for 30,000 timesteps.
Compare your printed **mean reward (last 10 episodes)** to Lab 16; values within roughly 10% are normal.

**Practical note:** actor-critic methods shine when action spaces are continuous or rewards are sparse. Here the comparison highlights *how* each algorithm learns, not a winner-take-all benchmark.
